# Multimodal Retrieval-Augmented AI System for Personalized Skincare Recommendations
This notebook implements a multi-agent RAG pipeline for generating personalized skincare
routine recommendations. The system accepts natural language user input, retrieves relevant
products from a vector database, checks for ingredient conflicts and allergens, and builds
a structured morning and evening skincare routine.

# Gemini API Setup & Test

In [ ]:
pip install google-genai

In [ ]:
from google import genai
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="What ingredients should someone with oily acne-prone skin look for in a cleanser?"
)
print(response.text)

For someone with oily, acne-prone skin, the goal of a cleanser is to effectively remove excess oil, makeup, and impurities without stripping the skin, while also helping to address acne concerns.

Here are the key ingredients and types of ingredients to look for:

**1. Active Acne-Fighting & Exfoliating Ingredients:**

*   **Salicylic Acid (BHA - Beta Hydroxy Acid):** This is often the **number one recommendation** for oily, acne-prone skin.
    *   **Why:** It's oil-soluble, meaning it can penetrate through oil and into the pores to exfoliate from within. This helps to dissolve dead skin cells and sebum that clog pores, preventing blackheads, whiteheads, and inflamed breakouts. It also has mild anti-inflammatory properties.
    *   **Look for:** Concentrations typically between 0.5% - 2% in cleansers.

*   **Benzoyl Peroxide:**
    *   **Why:** It's an antimicrobial agent that effectively kills *C. acnes* (formerly *P. acnes*), the bacteria that contributes to inflammatory acne. It al

# Data Cleaning & Merging

In [ ]:
!pip install chromadb google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import ast
import re

skincare_df = pd.read_csv("/content/drive/MyDrive/skincare_project/data/skincare_products_clean.csv")
cosmetics_df = pd.read_csv("/content/drive/MyDrive/skincare_project/data/cosmetics.csv")
inci_df = pd.read_csv("/content/drive/MyDrive/skincare_project/data/ingredientsList.csv")


In [ ]:
# Clean skincare_products_clean
# Define function that parses ingredient list from string representation of Python list
def parse_ingredients_list(ingred_str):
    try:
        ingredients = ast.literal_eval(ingred_str)
        return ", ".join([i.strip().lower() for i in ingredients])
    except (ValueError, SyntaxError, TypeError):
        return ""

# Apply cleaning to ingredient column
skincare_df["ingredients_clean"] = skincare_df["clean_ingreds"].apply(parse_ingredients_list)

# Standardize columns
skincare_df_clean = skincare_df[["product_name", "product_type", "ingredients_clean", "price"]].copy()
# Rename columns to match other dataset
skincare_df_clean.columns = ["name", "product_type", "ingredients", "price"]
# Add missing columns
skincare_df_clean["brand"] = "Unknown"
skincare_df_clean["source"] = "lookfantastic"
# Extract brand from product name
skincare_df_clean["brand"] = skincare_df_clean["name"].apply(lambda x: x.split(" ")[0] if pd.notna(x) else "Unknown")

# Add missing skin type columns  (not available in this dataset)
for col in ["combination", "dry", "normal", "oily", "sensitive"]:
    skincare_df_clean[col] = None

# Clean cosmetics dataset
cosmetics_df["ingredients_clean"] = cosmetics_df["Ingredients"].apply(
    lambda x: ", ".join([i.strip().lower() for i in str(x).split(",")]) if pd.notna(x) else ""
)

# Select and rename columns to match other dataset
cosmetics_df_clean = cosmetics_df[["Name", "Label", "ingredients_clean", "Price", "Brand",
                                    "Combination", "Dry", "Normal", "Oily", "Sensitive"]].copy()
cosmetics_df_clean.columns = ["name", "product_type", "ingredients", "price", "brand",
                               "combination", "dry", "normal", "oily", "sensitive"]
# Track source
cosmetics_df_clean["source"] = "sephora"

# Merge both datasets into one big table
products_df = pd.concat([skincare_df_clean, cosmetics_df_clean], ignore_index=True)

# Fix brand extraction for lookfantastic products
brand_mapping = {
    "The Ordinary": "The Ordinary",
    "CeraVe": "CeraVe",
    "AMELIORATE": "AMELIORATE",
    "La Roche-Posay": "La Roche-Posay",
    "Peter Thomas Roth": "Peter Thomas Roth",
    "Paula's Choice": "Paula's Choice",
    "Drunk Elephant": "Drunk Elephant",
}

# Define better brand extraction function
def extract_brand(name):
    if pd.isna(name):
        return "Unknown"
    # Try to match known brands
    for brand in brand_mapping:
        if brand.lower() in name.lower():
            return brand_mapping[brand]
    # Fallback: take first two words
    parts = name.split(" ")
    return " ".join(parts[:2]) if len(parts) >= 2 else parts[0]

# Only fix brands for lookfantastic products (Sephora already has brand column)
mask = products_df["source"] == "lookfantastic"
products_df.loc[mask, "brand"] = products_df.loc[mask, "name"].apply(extract_brand)

# Standardize product type names
product_type_mapping = {
    "Moisturiser": "Moisturizer", # UK --> US spelling
    "Eye Care": "Eye cream",
}

# Exclude non-facial products
exclude_types = ["Body Wash", "Bath Salts", "Bath Oil"]
print(f"Total before exclusion: {len(products_df)}")
products_df = products_df[~products_df["product_type"].isin(exclude_types)].reset_index(drop=True)
print(f"Total after excluding body products: {len(products_df)}")
products_df["product_type"] = products_df["product_type"].replace(product_type_mapping)

# Drop rows with no ingredients
products_df = products_df[products_df["ingredients"].str.len() > 0].reset_index(drop=True)

# Display first few rows of summary
print()
print(f"Total merged products: {len(products_df)}")
print(f"Product types: {products_df['product_type'].unique()}")
products_df.head()

Total before exclusion: 2610
Total after excluding body products: 2418
Total merged products: 2418
Product types: ['Moisturizer' 'Serum' 'Oil' 'Mist' 'Balm' 'Mask' 'Peel' 'Eye cream'
 'Cleanser' 'Toner' 'Exfoliator' 'Treatment' 'Face Mask' 'Sun protect']


,name,product_type,ingredients,price,brand,source,combination,dry,normal,oily,sensitive
0,The Ordinary Natural Moisturising Factors + HA...,Moisturizer,"capric triglyceride, cetyl alcohol, propanedio...",£5.20,The Ordinary,lookfantastic,None,None,None,None,None
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,Moisturizer,"homosalate, glycerin, octocrylene, ethylhexyl,...",£13.00,CeraVe,lookfantastic,None,None,None,None,None
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,Moisturizer,"sodium hyaluronate, sodium hyaluronate, panthe...",£6.20,The Ordinary,lookfantastic,None,None,None,None,None
3,AMELIORATE Transforming Body Lotion 200ml,Moisturizer,"ammonium lactate, c12-15, glycerin, prunus amy...",£22.50,AMELIORATE,lookfantastic,None,None,None,None,None
4,CeraVe Moisturising Cream 454g,Moisturizer,"glycerin, cetearyl alcohol, capric triglycerid...",£16.00,CeraVe,lookfantastic,None,None,None,None,None



# Build INCI ingredient knowledge base

In [ ]:
# Clean INCI data for use by conflict checker and retrieval
def clean_list_string(val):
    """Parse the string representation of list from INCI dataset"""
    if pd.isna(val):
        return ""
    try:
        items = ast.literal_eval(val)
        return ", ".join([i.strip() for i in items if i.strip()])
    except:
        return str(val).strip()

inci_df["good_for"] = inci_df["who_is_it_good_for"].apply(clean_list_string)
inci_df["avoid_for"] = inci_df["who_should_avoid"].apply(clean_list_string)

# Create text description for each ingredient (what gets embedded)
inci_df["full_description"] = inci_df.apply(lambda row:
    f"Ingredient: {row['name']}. "
    f"Description: {row['short_description'] if pd.notna(row['short_description']) else 'N/A'}. "
    f"Function: {row['what_does_it_do'] if pd.notna(row['what_does_it_do']) else 'N/A'}. "
    f"Good for: {row['good_for']}. "
    f"Avoid if: {row['avoid_for']}.",
    axis=1
)

print(f"Total ingredients in INCI database: {len(inci_df)}")
print(f"\nSample entry:\n{inci_df['full_description'].iloc[0][:500]}")

Total ingredients in INCI database: 248

Sample entry:
Ingredient: Alpha-Glucan Oligosaccharide. Description: Alpha-glucan oligosaccharide is in a class of prebiotic ingredients also found on ingredients lists as Fructooligosaccharides, lactobacillus extract filtrate, rhamnose and saccharomyces cerevisiae (yeast) extract. 

Prebiotics are a class of ingredients which can be found on ingredients lists as fructooligosaccharides, lactobacillus extract filtrate, rhamnose and saccharomyces cerevisiae (yeast) extract.. Function: Prebiotics offer benefit


# Create product documents for the vector database

In [ ]:
# Build text document for each product that captures all its info
# embedded and stored in ChromaDB
product_documents = []
product_metadata = []
product_ids = []

for idx, row in products_df.iterrows():
    # Build a text description
    skin_types = []
    for st in ["combination", "dry", "normal", "oily", "sensitive"]:
        if row.get(st) == 1:
            skin_types.append(st)
    skin_type_str = ", ".join(skin_types) if skin_types else "not specified"

    doc = (
        f"Product: {row['name']}. "
        f"Brand: {row['brand']}. "
        f"Type: {row['product_type']}. "
        f"Price: {row['price']}. "
        f"Suitable for skin types: {skin_type_str}. "
        f"Ingredients: {row['ingredients']}."
    )

    product_documents.append(doc)
    product_metadata.append({
        "name": str(row["name"]),
        "brand": str(row["brand"]),
        "product_type": str(row["product_type"]),
        "price": str(row["price"]),
        "source": str(row["source"]),
        "skin_types": skin_type_str
    })
    product_ids.append(f"product_{idx}")

print(f"Created {len(product_documents)} product documents")
print(f"\nSample document:\n{product_documents[0][:300]}")

Created 2418 product documents

Sample document:
Product: The Ordinary Natural Moisturising Factors + HA 30ml. Brand: The Ordinary. Type: Moisturizer. Price: £5.20. Suitable for skin types: not specified. Ingredients: capric triglyceride, cetyl alcohol, propanediol, stearyl alcohol, glycerin, sodium hyaluronate, arganine, aspartic acid, glycine, a


# Load into ChromaDB

In [ ]:
import chromadb

# Create ChromaDB client to convert text documents into vector embeddings
chroma_client = chromadb.Client()

# Delete old collection if exists, then create new one
try:
    chroma_client.delete_collection("skincare_products")
except:
    pass

# Create product collection with cosine similarity
product_collection = chroma_client.create_collection(
    name="skincare_products",
    metadata={"hnsw:space": "cosine"}
)

# Add documents in batches
batch_size = 500
for i in range(0, len(product_documents), batch_size):
    end = min(i + batch_size, len(product_documents))
    product_collection.add(
        documents=product_documents[i:end],
        metadatas=product_metadata[i:end],
        ids=product_ids[i:end]
    )
    print(f"Added batch {i//batch_size + 1}: products {i} to {end}")

print(f"\nTotal products in vector database: {product_collection.count()}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 26.9MiB/s]


Added batch 1: products 0 to 500
Added batch 2: products 500 to 1000
Added batch 3: products 1000 to 1500
Added batch 4: products 1500 to 2000
Added batch 5: products 2000 to 2418

Total products in vector database: 2418


In [ ]:
# Index INCI Ingredients into ChromaDB
try:
    chroma_client.delete_collection("skincare_ingredients")
except:
    pass

ingredient_collection = chroma_client.create_collection(
    name="skincare_ingredients",
    metadata={"hnsw:space": "cosine"}
)

# Create unique IDs for each ingredient
ingredient_ids = []
idx = 0
while idx < len(inci_df):
    ingredient_ids.append("ingredient_" + str(idx))
    idx = idx + 1

# Build metadata for filtering & display
ingredient_metadata = []
for idx, row in inci_df.iterrows():
    ingredient_metadata.append({
        "name": str(row["name"]) if pd.notna(row["name"]) else "",
        "good_for": str(row["good_for"]),
        "avoid_for": str(row["avoid_for"])
    })

# Index all ingredient descriptions into ChromaDB
ingredient_collection.add(
    documents=inci_df["full_description"].tolist(),
    metadatas=ingredient_metadata,
    ids=ingredient_ids
)

print("Total ingredients indexed: " + str(ingredient_collection.count()))

Total ingredients indexed: 248


# Agent 1: Skin Profile Agent

In [ ]:
import json

# Define function that takes raw natural language user input and extracts a structured skin profile using Gemini
def skin_profile_agent(user_input):
    prompt = f"""You are a skincare intake specialist. Analyze the following user input and extract
a structured skin profile. Return ONLY valid JSON with no markdown formatting, no backticks, no explanation.

User input: "{user_input}"

Return this exact JSON structure (use null for any field not mentioned):
{{
    "skin_type": "oily/dry/combination/sensitive/normal or null",
    "concerns": ["list of skin concerns mentioned, e.g., acne, dark circles, redness, wrinkles"],
    "allergies": ["list of any allergies or ingredients to avoid"],
    "age": null,
    "current_products": ["list of any products the user currently uses"],
    "goals": ["what the user wants to achieve, e.g., reduce acne, brighten skin"],
    "routine_request": "full routine / specific product / product suggestion"
}}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    try:
        # Parse Gemini's response as JSON
        profile = json.loads(response.text)
    except json.JSONDecodeError:
        # Try to extract JSON from response if there's extra text
        import re
        json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
        if json_match:
            profile = json.loads(json_match.group())
        else:
            profile = {"error": "Could not parse profile", "raw": response.text}

    return profile

# Test Agent 1
test_input = "I have oily acne-prone skin and I am allergic to fragrance. Looking for a full routine."
profile = skin_profile_agent(test_input)
print(json.dumps(profile, indent=2))

{
  "skin_type": "oily",
  "concerns": [
    "acne"
  ],
  "allergies": [
    "fragrance"
  ],
  "age": null,
  "current_products": [],
  "goals": [],
  "routine_request": "full routine"
}


# Agent 2: Product Retrieval Agent


In [ ]:
# Define function to search product vector database using natural language query
def search_products(query, n_results=5, product_type=None):

    # Build filter if product type is specified to return correct category of product
    filter = {"product_type": product_type} if product_type else None

    results = product_collection.query(
        query_texts=[query],
        n_results=n_results,
        where=filter
    )
    print(f"Query: '{query}'\n")
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        distance = results['distances'][0][i]
        print(f"  {i+1}. {meta['name']}")
        print(f"     Brand: {meta['brand']} | Type: {meta['product_type']} | Price: {meta['price']}")
        print(f"     Skin types: {meta['skin_types']}")
        print(f"     Relevance score: {1 - distance:.3f}")
        print()

# Test queries
search_products("gentle cleanser for oily acne-prone skin", product_type="Cleanser")
print("=" * 80)
search_products("moisturizer for dry sensitive skin without fragrance", product_type="Moisturizer")
print("=" * 80)
search_products("anti-aging serum with retinol", product_type="Serum")

Query: 'gentle cleanser for oily acne-prone skin'

  1. Make-Up Removing Cleansing Oil
     Brand: CAUDALIE | Type: Cleanser | Price: 28
     Skin types: combination, dry, normal, oily, sensitive
     Relevance score: 0.528

  2. EradiKate® Daily Cleanser Acne Treatment
     Brand: KATE SOMERVILLE | Type: Cleanser | Price: 38
     Skin types: combination, normal, oily
     Relevance score: 0.528

  3. Avène Cleanance Cleansing Gel 200ml
     Brand: Avène Cleanance | Type: Cleanser | Price: £12.00
     Skin types: not specified
     Relevance score: 0.524

  4. Acne Solutions Clarifying Lotion
     Brand: CLINIQUE | Type: Cleanser | Price: 17
     Skin types: combination, dry, normal, oily
     Relevance score: 0.520

  5. Cleanser
     Brand: EVE LOM | Type: Cleanser | Price: 80
     Skin types: combination, dry, normal, oily, sensitive
     Relevance score: 0.518

Query: 'moisturizer for dry sensitive skin without fragrance'

  1. Hydra-Essentiel Silky Cream - Normal to Dry Skin
     

# Agent 3: Conflict Checker

In [ ]:
# Defines ingredient interaction rules grounded in the Skincare Knowledge Graph dataset
# Rules are split into two categories:
#   - conflict_rules_raw: ingredient combinations to AVOID (adverse effects)
#   - beneficial_pairs: ingredient combinations that WORK WELL together

conflict_rules_raw = [
    {"ingredient_a": "retinol", "ingredient_b": "vitamin c", "type": "avoid", "reason": "Cancel out effects."},
    {"ingredient_a": "retinol", "ingredient_b": "aha", "type": "avoid", "reason": "Cancel out effects and cause irritation."},
    {"ingredient_a": "retinol", "ingredient_b": "bha", "type": "avoid", "reason": "May cause breakouts, dry skin, and irritation."},
    {"ingredient_a": "retinol", "ingredient_b": "citric acid", "type": "avoid", "reason": "Excessive dryness, redness, sensitivity, or a rash."},
    {"ingredient_a": "retinol", "ingredient_b": "benzoyl peroxide", "type": "avoid", "reason": "Too harsh for skin and cancel out effects."},
    {"ingredient_a": "aha", "ingredient_b": "niacinamide", "type": "avoid", "reason": "Can cause redness."},
    {"ingredient_a": "bha", "ingredient_b": "niacinamide", "type": "avoid", "reason": "Can cause redness."},
    {"ingredient_a": "aha", "ingredient_b": "vitamin c", "type": "avoid", "reason": "Can cause irritation."},
    {"ingredient_a": "bha", "ingredient_b": "vitamin c", "type": "avoid", "reason": "Can cause irritation."},
    {"ingredient_a": "salicylic acid", "ingredient_b": "benzoyl peroxide", "type": "avoid", "reason": "Can cause skin irritation."},
]

beneficial_pairs = [
    {"ingredient_a": "hyaluronic acid", "ingredient_b": "polyglutamic acid", "benefit": "Better hydration."},
    {"ingredient_a": "retinol", "ingredient_b": "niacinamide", "benefit": "Improves skin blemishes, diminishes ageing, and evens out skin tone."},
    {"ingredient_a": "retinol", "ingredient_b": "peptides", "benefit": "Produces collagen and hyaluronic acid, reduces inflammation, and increases cell turnover."},
    {"ingredient_a": "vitamin c", "ingredient_b": "vitamin e", "benefit": "Can help prevent photodamage."},
    {"ingredient_a": "vitamin c", "ingredient_b": "ferulic acid", "benefit": "Ferulic acid stabilizes vitamin C and fends off free radicals."},
]

# Build bidirectional conflict lookup so checking either ingredient finds the rule
conflict_lookup = {}

for rule in conflict_rules_raw:
    a = rule["ingredient_a"]
    b = rule["ingredient_b"]

    if a not in conflict_lookup:
        conflict_lookup[a] = []
    conflict_lookup[a].append({"conflicts_with": b, "type": "avoid", "reason": rule["reason"]})

    if b not in conflict_lookup:
        conflict_lookup[b] = []
    conflict_lookup[b].append({"conflicts_with": a, "type": "avoid", "reason": rule["reason"]})

# Build bidirectional beneficial lookup similarly
beneficial_lookup = {}

for pair in beneficial_pairs:
    a = pair["ingredient_a"]
    b = pair["ingredient_b"]

    if a not in beneficial_lookup:
        beneficial_lookup[a] = []
    beneficial_lookup[a].append({"pairs_with": b, "benefit": pair["benefit"]})

    if b not in beneficial_lookup:
        beneficial_lookup[b] = []
    beneficial_lookup[b].append({"pairs_with": a, "benefit": pair["benefit"]})

print("Conflict rules loaded for " + str(len(conflict_lookup)) + " ingredients")
print("Beneficial pairs loaded for " + str(len(beneficial_lookup)) + " ingredients")

Conflict rules loaded for 8 ingredients
Beneficial pairs loaded for 8 ingredients


# RAG Test: Combine ChromaDB retrieval with Gemini to generate a recommendations

In [ ]:
# NOTE: This is a standalone RAG test cell, not the full multi-agent pipeline.
# It does not use Agent 1 (Skin Profile) or Agent 3 (Conflict Checker).

from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# Define function to retrieve products and then create recommendation
def rag_recommend(user_query, n_results=5):

    # Retrieve relevant products
    results = product_collection.query(
        query_texts=[user_query],
        n_results=n_results
    )

    # Format retrieved products as context
    context = "RETRIEVED PRODUCTS FROM DATABASE:\n\n"
    for i in range(len(results['ids'][0])):
        meta = results['metadatas'][0][i]
        doc = results['documents'][0][i]
        context += f"Product {i+1}: {doc}\n\n"

    # Send to Gemini with the retrieved context
    prompt = f"""You are a skincare recommendation assistant. Based on the user's query and the
retrieved product information below, provide a personalized recommendation.

USER QUERY: {user_query}

{context}

Please recommend the most suitable products from the retrieved list. For each recommendation, explain:
1. Why this product suits the user's needs
2. Key beneficial ingredients for their concerns
3. Any ingredients to be cautious about
4. How to incorporate it into a routine (morning/evening, order of application)

If none of the retrieved products are a great fit, say so honestly."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

# Test basic RAG pipeline in isolation (without full multi-agent integration)
result = rag_recommend("I have oily skin with acne on my cheeks and some dark circles. I'm 22 and allergic to fragrance.")
print(result)

Here are the personalized recommendations based on your query and the retrieved product information:

### **Recommendation 1: Lightstim® for Acne**

1.  **Why this product suits your needs:**
    *   **Acne Treatment:** This device is specifically designed to treat acne using LED light therapy, making it a highly effective option for your cheek acne. Light therapy helps to reduce inflammation and kill acne-causing bacteria.
    *   **Oily Skin:** As a device, it introduces no topical ingredients, so it won't add oil or clog pores, making it perfectly suitable for oily skin.
    *   **Fragrance Allergy:** Being a device, it is inherently fragrance-free, which is crucial given your allergy.
    *   **Skin Type Suitability:** The product explicitly states it's suitable for "oily" and "sensitive" skin types, aligning well with your profile.

2.  **Key beneficial features for your concerns:**
    *   **LED Light Therapy:** It uses specific wavelengths of light (typically blue and red light)

# Full Pipeline: Agent 1 → Agent 2 → Agent 3 → Agent 4

# Evaluation

In [ ]:
test_cases = [
    {
        "id": "TC01",
        "description": "Oily acne-prone skin, fragrance allergy",
        "input": "I have oily acne-prone skin and I am allergic to fragrance. Looking for a full routine.",
        "expected_conflict": False,
        "expect_allergy_flag": True
    },
    {
        "id": "TC02",
        "description": "Conflict trigger: retinol and AHA combination",
        "input": "I want to use a retinol serum and an AHA exfoliant together for anti-aging and smoother skin.",
        "expected_conflict": True,
        "expect_allergy_flag": False
    },
    {
        "id": "TC03",
        "description": "Dry sensitive skin, minimal routine",
        "input": "I have very dry sensitive skin that gets red easily. I need a gentle cleanser and a heavy moisturizer.",
        "expected_conflict": False,
        "expect_allergy_flag": False
    }
]

eval_results = []

for tc in test_cases:
    print("Running: " + tc["id"] + " - " + tc["description"])
    result = full_pipeline(tc["input"])

    conflict_detected = len(result["conflict_report"]["conflicts"]) > 0
    allergy_flagged = len(result["conflict_report"]["allergy_flags"]) > 0

    eval_results.append({
        "test_id": tc["id"],
        "description": tc["description"],
        "conflict_detected": conflict_detected,
        "expected_conflict": tc["expected_conflict"],
        "conflict_correct": conflict_detected == tc["expected_conflict"],
        "allergy_flagged": allergy_flagged,
        "expected_allergy_flag": tc["expect_allergy_flag"],
        "allergy_correct": allergy_flagged == tc["expect_allergy_flag"]
    })
    print("")

eval_df = pd.DataFrame(eval_results)
print(eval_df[["test_id", "description", "conflict_correct", "allergy_correct"]])
print("\nConflict detection accuracy: " + str(eval_df["conflict_correct"].mean()))
print("Allergy detection accuracy: " + str(eval_df["allergy_correct"].mean()))

Running: TC01 - Oily acne-prone skin, fragrance allergy
AGENT 1: Extracting skin profile...
{
  "skin_type": "oily",
  "concerns": [
    "acne"
  ],
  "allergies": [
    "fragrance"
  ],
  "age": null,
  "current_products": [],
  "goals": [],
  "routine_request": "full routine"
}

AGENT 2: Retrieving products from vector database...
  Retrieved 2 products for: Cleanser
  Retrieved 2 products for: Toner
  Retrieved 2 products for: Serum
  Retrieved 2 products for: Moisturizer
  Retrieved 2 products for: Sun protect

AGENT 3: Checking ingredient conflicts...
  ✅ No conflicts found.

AGENT 4: Building personalized routine...
As an expert skincare routine builder, I've analyzed your skin profile (oily, acne concerns, fragrance allergy) and the provided products. Your allergy to fragrance is a critical factor, as many products, even those marketed as "natural" or "gentle," can contain sensitizing essential oils or synthetic fragrances.

Based on this, there are unfortunately some significan

TypeError: tuple indices must be integers or slices, not str